#### Gold star schema

Reshapes the clean silver tables into a dimensional STAR SCHEMA: conformed
dimensions with surrogate keys, fact tables keyed to them, an explicit
referential-integrity check (since the Lakehouse doesn't enforce FKs), and a
model-ready training table for the cost-overrun ML step -- with lineage
columns carried through so the train/test split is available at model time.

#### Cell 1

In [1]:
# Iports + config
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# calendar-safe parse + write (same settings silver needs)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# lineage columns that flow through from silver
LINEAGE = ["batch_id", "data_split"]

# load silver
s_proj  = spark.table("silver_projects")
s_sub   = spark.table("silver_subcontractors")
s_cost  = spark.table("silver_cost_line_items")
s_labor = spark.table("silver_labor_timesheets")
s_sched = spark.table("silver_schedule_tasks")
s_safe  = spark.table("silver_safety_incidents")
print("silver loaded:",
      {t: spark.table(f"silver_{t}").count() for t in
       ["projects","subcontractors","cost_line_items","labor_timesheets","schedule_tasks","safety_incidents"]})


StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 3, Finished, Available, Finished, False)

silver loaded: {'projects': 240, 'subcontractors': 110, 'cost_line_items': 2729, 'labor_timesheets': 6600, 'schedule_tasks': 2880, 'safety_incidents': 376}


#### Cell 2

In [2]:
# DIM_PROJECT (surrogate key + business attributes + lineage)
# Surrogate key via row_number() over a deterministic ordering. IMPORTANT: we
# do NOT use monotonically_increasing_id() -- that value can be recomputed by
# Spark and produce DIFFERENT ids when you join back to it, silently corrupting
# fact-to-dimension joins. row_number() over an ordered window is stable, and we
# materialize the dimension (cache) so the keys are fixed before facts join to it.
proj_window = Window.orderBy("project_id")

# Helper function necessary to clean the Dates for the Sematic model.
def clamp_date(colname):
    c = F.col(colname)
    return F.when((c >= F.lit("1990-01-01").cast(T.DateType())) &
                  (c <= F.lit("2035-12-31").cast(T.DateType())), c).otherwise(None)

dim_project = (s_proj
    .select("project_id", "project_name", "project_type", "region",
            "delivery_method", "contract_value", "contract_value_valid",
            "square_footage", "status", "start_date", "planned_end_date",
            "actual_end_date", "batch_id", "data_split")
    .withColumn("project_sk", F.row_number().over(proj_window))
    .withColumn("start_year",  F.year("start_date"))
    .withColumn("start_month", F.month("start_date"))
    .withColumn("is_winter_start", F.when(F.month("start_date").isin(12, 1, 2), True).otherwise(False))
    .withColumn("start_date",       clamp_date("start_date"))
    .withColumn("planned_end_date", clamp_date("planned_end_date"))
    .withColumn("actual_end_date",  clamp_date("actual_end_date"))
).cache()
dim_project.count()   # materialize so surrogate keys are fixed before any join
print(f"dim_project: {dim_project.count()} rows")


StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 4, Finished, Available, Finished, False)

dim_project: 240 rows


####  Cell 3

In [3]:
# DIM_VENDOR and DIM_DIVISION (deterministic surrogate keys)
dim_vendor = (s_sub
    .select("sub_id", "sub_name", "trade_focus", "region",
            "performance_rating", "prequalified")
    .withColumn("vendor_sk", F.row_number().over(Window.orderBy("sub_id")))
).cache()
dim_vendor.count()

dim_division = (s_cost
    .select("csi_division", "division_name").distinct()
    .withColumn("division_sk", F.row_number().over(Window.orderBy("csi_division")))
    .withColumn("division_group",
        F.when(F.col("csi_division").isin("21","22","23","26","27"), "MEP")
         .when(F.col("csi_division").isin("31","32","33","02"), "Sitework")
         .when(F.col("csi_division").isin("03","04","05"), "Structural")
         .otherwise("Other"))
).cache()
dim_division.count()
print(f"dim_vendor: {dim_vendor.count()} | dim_division: {dim_division.count()}")


StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 5, Finished, Available, Finished, False)

dim_vendor: 110 | dim_division: 17


#### Cell 4

In [4]:
# DIM_DATE (calendar spanning all dates in the data)
# Bound source dates to a sane window so a stray bad date (year 1, year 9999)
# can't blow up the calendar range or Power BI's DateTime transport limits.
LO_BOUND = F.lit("1990-01-01").cast(T.DateType())
HI_BOUND = F.lit("2035-12-31").cast(T.DateType())

date_cols = [
    s_proj.select(F.col("start_date").alias("d")),
    s_labor.select(F.col("work_date").alias("d")),
    s_safe.select(F.col("incident_date").alias("d")),
]
all_dates = (date_cols[0].union(date_cols[1]).union(date_cols[2])
             .filter(F.col("d").isNotNull())
             .filter((F.col("d") >= LO_BOUND) & (F.col("d") <= HI_BOUND)))   # <-- the fix
bounds = all_dates.agg(F.min("d").alias("lo"), F.max("d").alias("hi")).collect()[0]
lo, hi = bounds["lo"], bounds["hi"]

dim_date = (spark.sql(f"SELECT sequence(to_date('{lo}'), to_date('{hi}'), interval 1 day) AS ds")
    .select(F.explode("ds").alias("date"))
    .withColumn("date_sk", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("is_weekend", F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False))
)
print(f"dim_date: {dim_date.count()} days ({lo} to {hi})")

StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 6, Finished, Available, Finished, False)

dim_date: 1827 days (2021-08-09 to 2026-08-09)


#### Cell 5

In [5]:
# FELL 5 -- FACT tables (keyed to dimensions via surrogate keys)
proj_key = dim_project.select("project_id", "project_sk")
div_key  = dim_division.select("csi_division", "division_sk")

fact_cost = (s_cost
    .join(proj_key, "project_id", "left")
    .join(div_key, "csi_division", "left")
    .select("line_item_id", "project_sk", "division_sk",
            "budget_amount", "actual_amount", "change_order_amount",
            "variance", "overrun_ratio", "actual_valid",
            F.col("project_id"))   # keep natural key for traceability
)

fact_labor = (s_labor
    .join(proj_key, "project_id", "left")
    .withColumn("date_sk", F.date_format("work_date", "yyyyMMdd").cast("int"))
    .select("timesheet_id", "project_sk", "date_sk", "trade",
            "regular_hours", "overtime_hours", "hourly_rate", "total_cost",
            F.col("project_id"))
)

fact_safety = (s_safe
    .join(proj_key, "project_id", "left")
    .select("incident_id", "project_sk", "incident_type", "severity",
            "trade_involved", "lost_days", "is_lost_time", "root_cause",
            F.col("project_id"))
)

fact_schedule = (s_sched
    .join(proj_key, "project_id", "left")
    .select("task_id", "project_sk", "task_name", "planned_duration_days",
            "actual_duration_days", "duration_slip_days", "percent_complete",
            F.col("project_id"))
)
print(f"facts: cost={fact_cost.count()} labor={fact_labor.count()} "
      f"safety={fact_safety.count()} schedule={fact_schedule.count()}")


StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 7, Finished, Available, Finished, False)

facts: cost=2729 labor=6600 safety=376 schedule=2880


#### Cell 6

In [6]:
# REFERENTIAL INTEGRITY CHECK
# The Lakehouse does NOT enforce foreign keys, so we validate explicitly:
# every fact row's surrogate key must resolve to a dimension. Orphans (null
# project_sk after the left join) indicate a fact referencing a missing project.
print("=== REFERENTIAL INTEGRITY ===\n")
checks = {
    "fact_cost -> dim_project":     fact_cost.filter(F.col("project_sk").isNull()).count(),
    "fact_cost -> dim_division":    fact_cost.filter(F.col("division_sk").isNull()).count(),
    "fact_labor -> dim_project":    fact_labor.filter(F.col("project_sk").isNull()).count(),
    "fact_safety -> dim_project":   fact_safety.filter(F.col("project_sk").isNull()).count(),
    "fact_schedule -> dim_project": fact_schedule.filter(F.col("project_sk").isNull()).count(),
}
all_ok = True
for name, orphans in checks.items():
    status = "OK" if orphans == 0 else f"WARN: {orphans} orphaned rows"
    if orphans: all_ok = False
    print(f"  {name:32s} {status}")
print(f"\nReferential integrity: {'ALL PASSED' if all_ok else 'ORPHANS FOUND (investigate)'}")


StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 8, Finished, Available, Finished, False)

=== REFERENTIAL INTEGRITY ===

  fact_cost -> dim_project         OK
  fact_cost -> dim_division        OK
  fact_labor -> dim_project        OK
  fact_safety -> dim_project       OK
  fact_schedule -> dim_project     OK

Referential integrity: ALL PASSED


#### Cell 7

In [7]:
# Write star schema as Delta
gold = {
    "dim_project":   dim_project,
    "dim_vendor":    dim_vendor,
    "dim_division":  dim_division,
    "dim_date":      dim_date,
    "fact_cost":     fact_cost,
    "fact_labor":    fact_labor,
    "fact_safety":   fact_safety,
    "fact_schedule": fact_schedule,
}
for tbl, df in gold.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(tbl)
    print(f"wrote {tbl}")
    

StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 9, Finished, Available, Finished, False)

wrote dim_project
wrote dim_vendor
wrote dim_division
wrote dim_date
wrote fact_cost
wrote fact_labor
wrote fact_safety
wrote fact_schedule


#### Cell 8

In [8]:
# ML TRAINING TABLE (denormalized, model-ready, carries data_split)
# One row per cost line item with project attributes joined in -- exactly the
# grain the cost-overrun regressor trains on. Lineage columns come along so the
# model can filter train vs test by provenance.
ml_cost_training = (fact_cost
    .filter(F.col("actual_valid") & F.col("overrun_ratio").isNotNull())
    .filter(F.col("overrun_ratio").between(0.5, 3.0))   # drop injected outliers
    .join(dim_project.select("project_sk", "project_type", "delivery_method",
                             "region", "is_winter_start", "batch_id", "data_split"),
          "project_sk", "left")
    .join(dim_division.select("division_sk", "csi_division", "division_group"),
          "division_sk", "left")
    .withColumn("co_ratio",
        F.when(F.col("budget_amount") > 0,
               F.col("change_order_amount") / F.col("budget_amount")).otherwise(0.0))
    .select("line_item_id", "project_type", "delivery_method", "region",
            "csi_division", "division_group", "is_winter_start",
            "co_ratio", "overrun_ratio",          # <- target
            "batch_id", "data_split")
)
ml_cost_training.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ml_cost_training")
print(f"ml_cost_training: {ml_cost_training.count()} rows")

print("\n=== train/test split available at model time ===")
ml_cost_training.groupBy("data_split").agg(
    F.count("*").alias("rows"),
    F.round(F.mean("overrun_ratio"), 3).alias("avg_overrun")).show()

print("Gold layer complete: star schema + ml_cost_training ready for modeling.")


StatementMeta(, 22819be1-2e92-46b2-b2cc-615d45b0c3ad, 10, Finished, Available, Finished, False)

ml_cost_training: 2583 rows

=== train/test split available at model time ===
+----------+----+-----------+
|data_split|rows|avg_overrun|
+----------+----+-----------+
|     train|1304|      1.186|
|      test|1279|      1.199|
+----------+----+-----------+

Gold layer complete: star schema + ml_cost_training ready for modeling.
